# Sentiment Analysis System

## Project Overview

This project focuses on sentiment analysis using Natural Language Processing (NLP) and Machine Learning techniques. The objective is to classify text reviews into positive or negative sentiments based on the opinion expressed in the review.

The system analyzes customer feedback, product reviews, and user comments to determine whether the sentiment is:

- Positive
- Negative


## Objectives

- Perform sentiment classification on textual reviews.
- Clean and preprocess raw text data.
- Extract meaningful features using TF-IDF.
- Train and evaluate multiple machine learning models.
- Compare model performance.
- Build an ensemble model for improved prediction accuracy.
- Save the trained model for deployment in a web application.

## Technologies Used

### Programming Language

- Python

### Libraries

- Pandas
- NumPy
- Matplotlib
- Seaborn
- NLTK
- Scikit-Learn
- Transformers
- PyTorch
- Pickle


In [ ]:
import numpy as np
import pandas  as pd
import matplotlib.pyplot as plt
import seaborn as sns
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
import re

## Dataset Used

### Sentiment Analysis Dataset

The dataset contains textual reviews along with sentiment labels.

Typical dataset fields include:

- Review Text
- Sentiment Label

Label Conversion:

- 0 → Negative
- 1 → Positive


In [ ]:
df=pd.read_csv('/content/sentiment.csv',encoding='latin-1')

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 53154 entries, 0 to 53153
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   target  53154 non-null  int64 
 1   id      53154 non-null  int64 
 2   date    53154 non-null  object
 3   flag    53153 non-null  object
 4   user    53153 non-null  object
 5   text    53153 non-null  object
dtypes: int64(2), object(4)
memory usage: 2.4+ MB


## Data Preprocessing

### Text Cleaning

The following preprocessing steps were applied:

- Remove special characters
- Remove numbers
- Convert text to lowercase
- Remove extra spaces


In [ ]:
df.isnull().sum()

,0
target,0
id,0
date,0
flag,1
user,1
text,1


In [ ]:
df['target'].value_counts()

,count
target,
0,53154


In [ ]:
df['target']=df['target'].replace(4,1)

In [ ]:
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

### Stopword Removal

English stopwords were removed while preserving important negation words such as:

- no
- not
- nor
- never

This helps the model correctly understand phrases like:

- "not good"
- "not satisfied"
- "never recommend"

### Text Normalization

The text was normalized before feature extraction.


In [ ]:
import re
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

porter = PorterStemmer()

# Load stopwords once
stop_words = set(stopwords.words('english'))

# Remove negation words from stopwords
negation_words = {'no', 'not', 'nor', 'never'}
stop_words = stop_words - negation_words

def clean_text(content):
    content = re.sub('[^a-zA-Z]', ' ', content)
    content = content.lower()
    words = content.split()

    # Remove stopwords but keep negations
    words = [word for word in words if word not in stop_words]

    # Optional: comment this line if accuracy drops
    # words = [porter.stem(word) for word in words]

    return " ".join(words)

In [ ]:
df['steamed_test']=df['text'].apply(clean_text)

In [ ]:
df['steamed_test']

In [ ]:
x=df['steamed_test'].values
y=df['target'].values

In [ ]:
print(x)

In [ ]:
y

## Feature Engineering

### TF-IDF Vectorization

The project uses TF-IDF (Term Frequency–Inverse Document Frequency) to convert textual data into numerical vectors.

Configuration:

- N-grams: (1,2)
- Maximum Features: 50,000
- Minimum Document Frequency: 5
- Maximum Document Frequency: 0.9
- Sublinear TF Scaling


In [ ]:
tfidf = TfidfVectorizer(
    ngram_range=(1,2),      # unigrams + bigrams (VERY IMPORTANT)
    max_features=50000,     # large vocabulary for large dataset
    min_df=5,               # remove very rare noisy words
    max_df=0.9,             # remove extremely common words
    sublinear_tf=True       # improves performance on large corpora
)
x=tfidf.fit_transform(x)

In [ ]:
print(x)

In [ ]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test=train_test_split(x,y,stratify=y,test_size=0.2,random_state=42)

In [ ]:
print(x_train.shape)
print(x_test.shape)

In [ ]:
print(y_train.shape)
print(y_test.shape)

## Machine Learning Models Used

### Logistic Regression

A linear classification algorithm used for binary sentiment classification.

Features:

- Fast training
- High interpretability
- Strong baseline performance


In [ ]:
from sklearn.linear_model import LogisticRegression


In [ ]:
model=LogisticRegression(max_iter=1000)

In [ ]:
model.fit(x_train,y_train)

In [ ]:
y_pred=model.predict(x_test)

In [ ]:
from sklearn.metrics import accuracy_score
accuracy_score(y_test,y_pred)

In [ ]:
test_sentences = [
    "This product is absolutely amazing and works perfectly.",
    "This is the worst purchase I have ever made.",
    "This is not good at all.",
    "Best purchase ever... not.",
    "It works fine but could be better."
]

In [ ]:
for sen in test_sentences:
  tv=tfidf.transform([sen])
  predictt=model.predict(tv)
  print(sen)
  print(predictt)
  print('-----------------------------')

In [ ]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import VotingClassifier
from sklearn.metrics import accuracy_score,classification_report


## Machine Learning Models Used

### Logistic Regression

A linear classification algorithm used for binary sentiment classification.

Features:

- Fast training
- High interpretability
- Strong baseline performance

### Multinomial Naive Bayes

A probabilistic classifier commonly used for text classification tasks.

Features:

- Efficient on large datasets
- Performs well with TF-IDF features

### Support Vector Machine (Linear SVC)

A powerful classification algorithm capable of finding optimal decision boundaries.

Features:

- Effective for high-dimensional text data
- Strong generalization ability


## Ensemble Learning

To improve prediction accuracy, multiple models were combined using a Voting Classifier.

### Ensemble Components

- Logistic Regression
- Multinomial Naive Bayes
- Linear SVC

### Voting Method

Hard Voting

The final prediction is determined based on the majority vote from all models.

Benefits:

- Improved robustness
- Better generalization
- Reduced model bias


In [ ]:
lr=LogisticRegression(C=0.5,solver='saga',max_iter=2000)
nb=MultinomialNB()
svc=LinearSVC(C=1)

ensebler_model=VotingClassifier(
    estimators=[('lr',lr),('svm',svc),('nm',nb)],
    voting='hard'
)

ensebler_model.fit(x_train,y_train)
y_pred=ensebler_model.predict(x_test)
print(accuracy_score(y_test,y_pred))
print(classification_report(y_test,y_pred))

## Testing and Prediction

The trained models were tested on custom review samples such as:

- "I absolutely love this product."
- "This is terrible and useless."
- "The product is okay but not great."
- "Best purchase ever... not."

The system successfully predicts the sentiment of unseen text data.


In [ ]:
test_sentences = [
    "I absolutely love this product",
    "This is terrible and useless",
    "The product is okay but not great",
    "Great, it broke again",
    "Fantastic service if you enjoy waiting forever"
]


## Model Evaluation

The following metrics were used to evaluate model performance:

- Accuracy Score
- Precision
- Recall
- F1-Score
- Classification Report

These metrics help assess how effectively the model identifies positive and negative sentiments.


In [ ]:
for sen in test_sentences:
  tv=tfidf.transform([sen])
  predictt=model.predict(tv)
  print(sen)
  print(predictt)
  print('-----------------------------')

In [ ]:
!pip install transformers torch datasets

In [ ]:
import pandas as pd
from transformers import pipeline

In [ ]:
classifier = pipeline("sentiment-analysis")

In [ ]:
classifier = pipeline("sentiment-analysis")

In [ ]:
result = classifier("Best purchase ever... not")
print(result)

[{'label': 'POSITIVE', 'score': 0.9991299510002136}]


## Expected Outcome

The final system automatically analyzes textual reviews and accurately determines whether the sentiment expressed is positive or negative. This enables businesses and organizations to gain insights from customer feedback and make data-driven decisions.


In [ ]:
sentences = [
    "This product is amazing",
    "Worst purchase ever",
    "It works fine but could be better",
    "Best purchase ever... not"
]

for s in sentences:
    print(s)
    print(classifier(s))
    print("-----------------")

This product is amazing
[{'label': 'POSITIVE', 'score': 0.9998852014541626}]
-----------------
Worst purchase ever
[{'label': 'NEGATIVE', 'score': 0.9997972846031189}]
-----------------
It works fine but could be better
[{'label': 'POSITIVE', 'score': 0.9869810938835144}]
-----------------
Best purchase ever... not
[{'label': 'POSITIVE', 'score': 0.9991299510002136}]
-----------------


In [ ]:
sentences = [
    "I absolutely love this product",
    "This is terrible and useless",
    "The product is okay but not great",
    "Great, it broke again",
    "Fantastic service if you enjoy waiting forever"
]

for s in sentences:
    print(s)
    print(classifier(s))
    print("----------------")

I absolutely love this product
[{'label': 'POSITIVE', 'score': 0.9998824596405029}]
----------------
This is terrible and useless
[{'label': 'NEGATIVE', 'score': 0.999805748462677}]
----------------
The product is okay but not great
[{'label': 'NEGATIVE', 'score': 0.9885303378105164}]
----------------
Great, it broke again
[{'label': 'POSITIVE', 'score': 0.9998067021369934}]
----------------
Fantastic service if you enjoy waiting forever
[{'label': 'POSITIVE', 'score': 0.9998456239700317}]
----------------


In [ ]:
import pickle as pkl

In [ ]:
pkl.dump(ensebler_model,open('senti.pkl','wb'))

In [ ]:
pkl.dump(tfidf,open('tfidf.pkl','wb'))